In [3]:
!jupyter nbconvert --to html Downloads/PhoBERT_train.ipynb




This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr

[NbConvertApp] WARNING | pattern 'Downloads/PhoBERT_train.ipynb' matched no files


In [ ]:
!pip install vncorenlp


In [ ]:
# Update pip
!pip install --upgrade pip setuptools wheel

# Torch + Transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install "transformers>=4.38"

# ML/DL packages
!pip install scikit-learn pandas numpy tqdm

# VnCoreNLP
!pip install git+https://github.com/vncorenlp/VnCoreNLP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


Looking in indexes: https://download.pytorch.org/whl/cpu
  Cloning https://github.com/vncorenlp/VnCoreNLP.git to /tmp/pip-req-build-921t7yxm
  Running command git clone --filter=blob:none --quiet https://github.com/vncorenlp/VnCoreNLP.git /tmp/pip-req-build-921t7yxm
  Resolved https://github.com/vncorenlp/VnCoreNLP.git to commit 62bbc58fe5d113c898eae112656be97dcf50b3a0
ERROR: git+https://github.com/vncorenlp/VnCoreNLP.git does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
import os, math, re, warnings, numpy as np, pandas as pd, torch, torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
# --- THAY ĐỔI 1: Cập nhật các hàm metrics được import ---
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             precision_score, recall_score)
import string

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

# ====================== CẤU HÌNH ======================
TRAIN_CSV = "/content/train_phobert_multitask.csv"
TEST_CSV  = "/content/test_phobert_multitask.csv"

TICKER_FILES = [
    "/content/Macp .xlsx",
]

MODEL_NAME = "vinai/phobert-large"
EPOCHS = 40
BATCH_SIZE = 32
LR = 2e-5
MAX_LENGTH = 256
USE_WEIGHTED_SAMPLER = True
USE_SOFT_LABELS = True
USE_VNCORE = True
USE_ENTITY_MASK = True
WARMUP_RATIO = 0.06
PATIENCE = 14
SEED = 42
USE_TEXT_NORMALIZATION = True  # Thêm flag để bật/tắt chuẩn hóa văn bản

# ====================== CHUẨN HÓA VĂN BẢN ======================
def normalize_text(text: str) -> str:
    """
    Chuẩn hóa văn bản: chuyển chữ thường và loại bỏ dấu câu
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Chuyển sang chữ thường
    text = text.lower()

    # Loại bỏ dấu câu (giữ lại khoảng trắng)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Loại bỏ nhiều khoảng trắng liên tiếp
    text = re.sub(r'\s+', ' ', text)

    # Loại bỏ khoảng trắng đầu/cuối
    text = text.strip()

    return text

# ====================== VnCoreNLP (tách từ) ======================
def get_vncorenlp():
    from vncorenlp import VnCoreNLP
    jar = "vncorenlp/VnCoreNLP-1.1.1.jar"
    if not os.path.exists(jar):
        os.system("mkdir -p vncorenlp/models/wordsegmenter")
        os.system("wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.1.1.jar -O vncorenlp/VnCoreNLP-1.1.1.jar")
        os.system("wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab -O vncorenlp/models/wordsegmenter/vi-vocab")
        os.system("wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr -O vncorenlp/models/wordsegmenter/wordsegmenter.rdr")
    return VnCoreNLP(jar, annotators="wseg", max_heap_size='-Xmx2g')

def vn_word_segment(rdr, s: str) -> str:
    if not isinstance(s, str) or not s.strip(): return ""
    sents = rdr.tokenize(s)
    return " ".join(" ".join(tokens) for tokens in sents)

# ====================== Entity masking (PER/ORG/LOC/TICKER) ======================
def load_ticker_set():
    tickers = set()
    for p in TICKER_FILES:
        if os.path.exists(p):
            try:
                df_t = pd.read_excel(p)
                for col in df_t.columns:
                    s = df_t[col]
                    vals = s.dropna().astype(str).str.strip().unique().tolist()
                    for v in vals:
                        if re.fullmatch(r"[A-Z]{2,6}", v):
                            tickers.add(v)
            except Exception as e:
                print("⚠️ Không đọc được", p, e)
    if tickers:
        print(f"Loaded {len(tickers)} tickers từ file.")
    return tickers

TICKER_SET = load_ticker_set()

def build_ticker_regex(ticker_set):
    if not ticker_set: return None
    escaped = [re.escape(t) for t in sorted(ticker_set, key=len, reverse=True)]
    pat = r"\b(" + "|".join(escaped) + r")\b"
    return re.compile(pat)

TICKER_RE = build_ticker_regex(TICKER_SET)

try:
    from underthesea import ner as uts_ner
    UTS_OK = True
except Exception:
    print("⚠️ underthesea chưa cài — tắt entity masking.")
    UTS_OK = False
    USE_ENTITY_MASK = False

def mask_entities(text: str) -> str:
    if not text: return text
    s = text
    if TICKER_RE is not None:
        s = TICKER_RE.sub("<NAME>", s)
    if UTS_OK:
        try:
            tags = uts_ner(s)
            out_tokens, i = [], 0
            while i < len(tags):
                token_info = tags[i]; token = token_info[0]; ner_tag = token_info[-1]
                if str(ner_tag).startswith("B-"):
                    ent_type = ner_tag[2:]
                    j = i + 1
                    while j < len(tags) and str(tags[j][-1]).startswith("I-"): j += 1
                    if ent_type in ("PER","ORG"):
                        out_tokens.append("<NAME>")
                    elif ent_type == "LOC":
                        out_tokens.append("<LOC>")
                    else:
                        out_tokens.append(token)
                    i = j
                else:
                    if re.fullmatch(r"[A-Z]{2,6}", token):
                        out_tokens.append("<NAME>")
                    else:
                        out_tokens.append(token)
                    i += 1
            s = " ".join(out_tokens)
        except Exception:
            pass
    return s

# ====================== LOAD DỮ LIỆU (GIỮ NGUYÊN, có nhớ tên cột gốc) ======================
ORIG_TEXT_COL = None
RAW_TEST_DF = None # Thêm biến để lưu DataFrame test gốc

def load_csv(path):
    global ORIG_TEXT_COL, RAW_TEST_DF
    df = pd.read_csv(path)
    text_col = None
    for c in df.columns:
        cl = str(c).lower()
        if cl in ["text", "lọc", "loc"]:
            text_col = c; break
    assert text_col is not None, f"Thiếu cột văn bản (text/ lọc/ loc). Thấy: {df.columns.tolist()}"

    if path == TRAIN_CSV:
        ORIG_TEXT_COL = text_col

    if path == TEST_CSV:
        RAW_TEST_DF = df.copy()

    assert "llm_sentiment" in df.columns and "llm_risk" in df.columns, "Cần cột llm_sentiment và llm_risk"
    df = df[["llm_sentiment","llm_risk",text_col]].copy()
    df.rename(columns={text_col:"text"}, inplace=True)
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len()>0]
    df["llm_sentiment"] = pd.to_numeric(df["llm_sentiment"], errors="coerce").astype("Int64")
    df["llm_risk"] = pd.to_numeric(df["llm_risk"], errors="coerce").astype("Int64")
    df = df[df["llm_sentiment"].between(1,5, inclusive="both") & df["llm_risk"].between(1,5, inclusive="both")]
    df = df.dropna(subset=["llm_sentiment","llm_risk"])
    return df

train_df = load_csv(TRAIN_CSV)
test_df  = load_csv(TEST_CSV)
print(f"Loaded: train={len(train_df)}, test={len(test_df)}")

# ====================== BƯỚC 1: CHUẨN HÓA VĂN BẢN ======================
if USE_TEXT_NORMALIZATION:
    print("• Bước 1: Chuẩn hóa văn bản (chuyển chữ thường + loại bỏ dấu câu) ...")

    # Lưu text gốc trước khi chuẩn hóa (để xuất ra file Excel sau này)
    train_df["text_original"] = train_df["text"].copy()
    test_df["text_original"] = test_df["text"].copy()

    # Áp dụng chuẩn hóa
    train_df["text"] = train_df["text"].apply(normalize_text)
    test_df["text"] = test_df["text"].apply(normalize_text)

    # Loại bỏ các dòng có text rỗng sau khi chuẩn hóa
    train_df = train_df[train_df["text"].str.len() > 0]
    test_df = test_df[test_df["text"].str.len() > 0]

    print(f"  Sau chuẩn hóa: train={len(train_df)}, test={len(test_df)}")

# ====================== BƯỚC 2: ENTITY MASKING ======================
if USE_ENTITY_MASK:
    print("• Bước 2: Áp dụng entity masking ...")
    train_df["text"] = train_df["text"].map(mask_entities)
    test_df["text"]  = test_df["text"].map(mask_entities)

# ====================== BƯỚC 3: TÁCH TỪ ======================
if USE_VNCORE:
    print("• Bước 3: Tách từ với VnCoreNLP ...")
    rdr = get_vncorenlp()
    train_df["text"] = train_df["text"].astype(str).apply(lambda s: vn_word_segment(rdr, s))
    test_df["text"]  = test_df["text"].astype(str).apply(lambda s: vn_word_segment(rdr, s))

# ====================== DATASET (LÀM CỨNG SOFT-LABELS) ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class NewsDataset(Dataset):
    def __init__(self, df, max_length=384, use_soft_labels=False, main=0.8, neigh=0.1):
        self.text = df["text"].tolist()
        self.y_sent = (df["llm_sentiment"].astype(int) - 1).tolist()  # 0..4
        self.y_risk = (df["llm_risk"].astype(int) - 1).tolist()
        self.max_length = max_length
        self.use_soft = use_soft_labels
        self.main = main; self.neigh = neigh

    def __len__(self): return len(self.text)

    def _soft(self, c, C=5):
        t = torch.zeros(C, dtype=torch.float)
        t[c] = self.main
        if c-1 >= 0: t[c-1] = self.neigh
        if c+1 < C: t[c+1] = self.neigh
        return t

    def __getitem__(self, i):
        enc = tokenizer(
            self.text[i], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        item = {k:v.squeeze(0) for k,v in enc.items()}
        ys, yr = self.y_sent[i], self.y_risk[i]
        if self.use_soft:
            item["labels_sent_soft"] = self._soft(ys)
            item["labels_risk_soft"] = self._soft(yr)
        else:
            item["labels_sent"] = torch.tensor(ys, dtype=torch.long)
            item["labels_risk"] = torch.tensor(yr, dtype=torch.long)
        return item

ds_tr = NewsDataset(train_df, max_length=MAX_LENGTH, use_soft_labels=USE_SOFT_LABELS, main=0.8, neigh=0.1)
ds_te = NewsDataset(test_df , max_length=MAX_LENGTH, use_soft_labels=False)

# ====================== CLASS WEIGHTS + SAMPLER ======================
ytr_s = (train_df["llm_sentiment"].astype(int)-1).to_numpy()
ytr_r = (train_df["llm_risk"].astype(int)-1).to_numpy()
classes = np.arange(5)
w_s = compute_class_weight("balanced", classes=classes, y=ytr_s)
w_r = compute_class_weight("balanced", classes=classes, y=ytr_r)
w_s_t = torch.tensor(w_s, dtype=torch.float, device=device)
w_r_t = torch.tensor(w_r, dtype=torch.float, device=device)

if USE_WEIGHTED_SAMPLER:
    per_sample_w = (w_s[ytr_s] * w_r[ytr_r])
    sampler = WeightedRandomSampler(weights=per_sample_w, num_samples=len(per_sample_w), replacement=True)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, pin_memory=True)
else:
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

# ====================== MÔ HÌNH ======================
class PhoBertMultiHead(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=5, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head_sent = nn.Linear(h, num_labels)
        self.head_risk = nn.Linear(h, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:,0,:]
        pooled = self.dropout(pooled)
        return self.head_sent(pooled), self.head_risk(pooled)

model = PhoBertMultiHead().to(device)

# ====================== LOSS/OPT/SCHED ======================
if USE_SOFT_LABELS:
    loss_fn_sent = nn.KLDivLoss(reduction="batchmean")
    loss_fn_risk = nn.KLDivLoss(reduction="batchmean")
else:
    loss_fn_sent = nn.CrossEntropyLoss(weight=w_s_t)
    loss_fn_risk = nn.CrossEntropyLoss(weight=w_r_t)

opt = AdamW(model.parameters(), lr=LR)
total_steps = EPOCHS * math.ceil(len(ds_tr)/BATCH_SIZE)
warmup_steps = int(WARMUP_RATIO * total_steps)
sched = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ====================== EVAL ======================
# --- THAY ĐỔI 2: Cập nhật hoàn toàn hàm eval_on_loader ---
def eval_on_loader(dl):
    model.eval()
    yps_s, yts_s = [], []
    yps_r, yts_r = [], []
    with torch.no_grad():
        for b in dl:
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            logit_s, logit_r = model(ids, mask)
            yps_s.extend(logit_s.argmax(-1).cpu().numpy())
            yps_r.extend(logit_r.argmax(-1).cpu().numpy())
            if "labels_sent" in b:
                yts_s.extend(b["labels_sent"].cpu().numpy())
                yts_r.extend(b["labels_risk"].cpu().numpy())
            else:
                yts_s.extend(b["labels_sent_soft"].argmax(-1).cpu().numpy())
                yts_r.extend(b["labels_risk_soft"].argmax(-1).cpu().numpy())

    # Tính toán các metrics cho Sentiment
    acc_s = accuracy_score(yts_s, yps_s)
    prec_s = precision_score(yts_s, yps_s, average="macro", zero_division=0)
    rec_s = recall_score(yts_s, yps_s, average="macro", zero_division=0)
    f1m_s = f1_score(yts_s, yps_s, average="macro", zero_division=0)
    f1c_s = f1_score(yts_s, yps_s, average=None, zero_division=0).tolist()
    qwk_s = cohen_kappa_score(np.array(yts_s)+1, np.array(yps_s)+1, weights="quadratic")

    # Tính toán các metrics cho Risk
    acc_r = accuracy_score(yts_r, yps_r)
    prec_r = precision_score(yts_r, yps_r, average="macro", zero_division=0)
    rec_r = recall_score(yts_r, yps_r, average="macro", zero_division=0)
    f1m_r = f1_score(yts_r, yps_r, average="macro", zero_division=0)
    f1c_r = f1_score(yts_r, yps_r, average=None, zero_division=0).tolist()
    qwk_r = cohen_kappa_score(np.array(yts_r)+1, np.array(yps_r)+1, weights="quadratic")

    return {
        "sent": {"acc": acc_s, "prec": prec_s, "rec": rec_s, "f1m": f1m_s, "f1c": f1c_s, "qwk": qwk_s},
        "risk": {"acc": acc_r, "prec": prec_r, "rec": rec_r, "f1m": f1m_r, "f1c": f1c_r, "qwk": qwk_r}
    }

# ====================== TRAIN ======================
best_score = -1.0
pat_left = PATIENCE
best_path = "/content/best_phobert_multitask.pt"

for ep in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    for b in tqdm(dl_tr, desc=f"Epoch {ep}/{EPOCHS}"):
        ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logit_s, logit_r = model(ids, mask)
            if USE_SOFT_LABELS and "labels_sent_soft" in b:
                ls = b["labels_sent_soft"].to(device)
                lr = b["labels_risk_soft"].to(device)
                loss_s = nn.functional.kl_div(nn.functional.log_softmax(logit_s, dim=-1), ls, reduction="batchmean")
                loss_r = nn.functional.kl_div(nn.functional.log_softmax(logit_r, dim=-1), lr, reduction="batchmean")
            else:
                ys = b["labels_sent"].to(device)
                yr = b["labels_risk"].to(device)
                loss_s = loss_fn_sent(logit_s, ys)
                loss_r = loss_fn_risk(logit_r, yr)
            loss = 0.5*(loss_s+loss_r)

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item()

    m = eval_on_loader(dl_te)
    print(f"\n[Epoch {ep}] train_loss={running/len(dl_tr):.4f}")
    # --- THAY ĐỔI 3: Cập nhật các câu lệnh print ---
    print(f"Sentiment: acc={m['sent']['acc']:.4f} prec={m['sent']['prec']:.4f} rec={m['sent']['rec']:.4f} f1_macro={m['sent']['f1m']:.4f} qwk={m['sent']['qwk']:.4f}")
    print("  F1 per-class (1..5):", [round(x,3) for x in m['sent']['f1c']])
    print(f"Risk     : acc={m['risk']['acc']:.4f} prec={m['risk']['prec']:.4f} rec={m['risk']['rec']:.4f} f1_macro={m['risk']['f1m']:.4f} qwk={m['risk']['qwk']:.4f}")
    print("  F1 per-class (1..5):", [round(x,3) for x in m['risk']['f1c']])

    score = 0.5*(m['sent']['qwk'] + m['risk']['qwk'])
    if score > best_score:
        best_score = score; pat_left = PATIENCE
        torch.save({"state_dict": model.state_dict(), "model_name": MODEL_NAME,
                    "epoch": ep, "best_score": best_score}, best_path)
        print(f"  ✓ New best={best_score:.4f} → saved to {best_path}")
    else:
        pat_left -= 1
        print(f"  no improve, patience_left={pat_left}")
        if pat_left <= 0:
            print("  Early stopping.")
            break

print(f"\nDone. Best macro-F1 mean on TEST: {best_score:.4f}")
print(f"Best model path: {best_path}")

Device: GPU
Loaded 1450 tickers từ file.
⚠️ underthesea chưa cài — tắt entity masking.
Loaded: train=1595, test=532
• Bước 1: Chuẩn hóa văn bản (chuyển chữ thường + loại bỏ dấu câu) ...
  Sau chuẩn hóa: train=1595, test=532
• Bước 3: Tách từ với VnCoreNLP ...


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


[Epoch 31] train_loss=-0.0396
Sentiment: acc=0.6429 prec=0.5794 rec=0.5727 f1_macro=0.5743 qwk=0.7361
  F1 per-class (1..5): [0.554, 0.667, 0.699, 0.561, 0.39]
Risk     : acc=0.5489 prec=0.5562 rec=0.4960 f1_macro=0.5123 qwk=0.6921
  F1 per-class (1..5): [0.425, 0.543, 0.576, 0.566, 0.452]
  no improve, patience_left=2


Epoch 32/40: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]



[Epoch 32] train_loss=-0.0392
Sentiment: acc=0.6466 prec=0.5835 rec=0.5915 f1_macro=0.5831 qwk=0.7428
  F1 per-class (1..5): [0.563, 0.667, 0.697, 0.588, 0.4]
Risk     : acc=0.5489 prec=0.5223 rec=0.5008 f1_macro=0.5055 qwk=0.6897
  F1 per-class (1..5): [0.419, 0.543, 0.578, 0.575, 0.412]
  no improve, patience_left=1


Epoch 33/40: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]



[Epoch 33] train_loss=-0.0435
Sentiment: acc=0.6391 prec=0.5689 rec=0.5782 f1_macro=0.5715 qwk=0.7445
  F1 per-class (1..5): [0.551, 0.673, 0.688, 0.566, 0.381]
Risk     : acc=0.5564 prec=0.5257 rec=0.5029 f1_macro=0.5083 qwk=0.6953
  F1 per-class (1..5): [0.41, 0.546, 0.592, 0.582, 0.412]
  no improve, patience_left=0
  Early stopping.

Done. Best macro-F1 mean on TEST: 0.7220
Best model path: /content/best_phobert_multitask.pt
